# Anti-Patterns & Pattern Selection

## What's covered

- Six common ways patterns fail in practice — **god class**, **singleton abuse**, **anemic domain model**, **pattern fever**, **lava layers**, **premature generalization**
- A small set of **code smells** that reliably suggest a specific pattern is the right answer
- A **decision framework** for picking patterns based on smells that have actually appeared — not on speculation
- The **rule of three** and why it beats "design for the future"
- When to *delete* a pattern that no longer earns its keep


## The premise

Six notebooks of pattern-by-pattern coverage have a built-in failure mode: they make patterns look like the answer. They are not. **The default is no pattern.** The simplest code that solves today's problem is the right starting point. Patterns enter when a specific smell appears that they specifically address.

This notebook is the counter-weight. The most common reason real codebases become hard to work with is not "they forgot to use the Visitor pattern." It is "they over-applied patterns they did not need, or they reached for the wrong one." Pattern *vocabulary* is valuable. Pattern *ceremony* is expensive. Knowing which to apply, when, and — equally importantly — when to remove a pattern that has outlived its usefulness, is what separates engineers who use patterns well from those who use them religiously.

We cover six anti-patterns by name, then turn to the decision framework.


## Anti-pattern 1 — God Class

**The smell:** one class does everything. Hundreds or thousands of lines. Methods for computing data, formatting it, persisting it, sending notifications, logging metrics, and applying business rules. Every change touches this file. Every team has it in their PR queue.

**Why it happens:** new functionality gets added where the existing functionality lives, because that's the path of least resistance. Each addition is "just one more method." The class accretes responsibilities one commit at a time.

**The cost:** the class violates the Single Responsibility Principle by an order of magnitude. Tests touch every dependency in the system because the class touches everything. Merge conflicts on this file are constant. Onboarding documentation calls it "the main class."

**The fix:** decompose by *actor*. Identify the distinct stakeholders who would request a change — the formatting team, the storage team, the business-rules team — and split the class along those lines. The pieces become injectable dependencies. The original class either disappears or becomes a thin coordinator.

**The patterns that help:** Single Responsibility Principle (notebook 01), Facade if you still want a single entry point (notebook 03), Strategy / Template Method for the algorithm pieces (notebook 04).


### Example — splitting a god class


In [ ]:
# Before — a god class for reports.
class ReportGod:
    def __init__(self, rows): self.rows = rows
    def total(self): return sum(r["amount"] for r in self.rows)
    def average(self): return self.total() / max(len(self.rows), 1)
    def to_html(self): return "<table>" + "".join(f"<tr><td>{r['name']}</td></tr>" for r in self.rows) + "</table>"
    def to_csv(self): return "\n".join(f"{r['name']},{r['amount']}" for r in self.rows)
    def save_to_disk(self, path, fmt): open(path, "w").write(self.to_html() if fmt == "html" else self.to_csv())
    def email_to(self, address, fmt): print(f"  sent {fmt} report to {address}")
    def log_access(self, user): print(f"  [audit] {user} accessed report")
    # ...and twenty more methods


# After — decomposed by actor. Each class has one reason to change.
from dataclasses import dataclass


@dataclass
class Report:
    rows: list[dict]
    def total(self) -> float: return sum(r["amount"] for r in self.rows)
    def average(self) -> float: return self.total() / max(len(self.rows), 1)


class HtmlRenderer:
    def render(self, r: Report) -> str:
        return "<table>" + "".join(f"<tr><td>{x['name']}</td></tr>" for x in r.rows) + "</table>"


class CsvRenderer:
    def render(self, r: Report) -> str:
        return "\n".join(f"{x['name']},{x['amount']}" for x in r.rows)


class FileStore:
    def save(self, path: str, content: str) -> None:
        open(path, "w").write(content)


class EmailService:
    def send(self, address: str, content: str) -> None:
        print(f"  sent report to {address}")


class AuditLog:
    def record_access(self, user: str) -> None:
        print(f"  [audit] {user} accessed report")


# The "Report" object stays tiny. Composition wires the rest.
r = Report(rows=[{"name": "alice", "amount": 100}, {"name": "bob", "amount": 50}])
html = HtmlRenderer().render(r)
FileStore().save("/tmp/r.html", html)
EmailService().send("ops@x", html)
AuditLog().record_access("alice")
print(f"total: {r.total()}")


The decomposed version is six small classes instead of one big one. Each has a clear actor. Each can be tested without dragging in the others. Crucially, you can add a *seventh* concern — a PDF renderer, a Slack notifier, an S3 store — without touching any of the existing six.


## Anti-pattern 2 — Singleton Abuse

**The smell:** a class is a Singleton because "we only need one for now" or "passing it everywhere is annoying." Tests can't run in parallel because they share state through it. Mocking it requires monkey-patching. Two parts of the codebase that should not know about each other are coupled through the singleton.

**Why it happens:** Singleton is the easiest GoF pattern to apply (one line in Kotlin, a few in Python) and it *feels* like a clean solution to "I need a logger / config / connection pool, everywhere." It avoids the perceived hassle of threading dependencies through constructors.

**The cost:** the singleton is global state with one extra layer of indirection. Every problem global state has — implicit coupling, untestability, surprising mutation — applies to it.

**The fix:** make the dependency explicit. Pass it through the constructor. Use a composition root that constructs exactly one instance and passes it everywhere it's needed. If wiring becomes painful, *then* introduce an IoC container — not a singleton.

**The legitimate uses:** the object's uniqueness is part of its identity — a process-wide log handler that everyone agrees to, a hardware device handle, a pre-warmed connection pool. The litmus test: *can you imagine wanting two of these in the same process for tests?* If yes, it's not a singleton.


## Anti-pattern 3 — Anemic Domain Model

**The smell:** domain objects are just bags of data with getters and setters. All the actual behaviour lives in "service" or "manager" classes that operate on the data. `Order` has fields and accessors; `OrderService` has every method that does anything to an order.

**Why it happens:** popular in codebases that started from database schemas and grew layered architectures around them. "Models are data, services are logic" feels orderly, especially when the framework or the ORM encourages it. Java's "JavaBeans" tradition pushed in this direction for two decades.

**The cost:** behaviour and the state it operates on live in different files. Reading the domain is harder — to understand what an `Order` can do, you read the field list, then track down every service that touches an `Order`. Encapsulation is weakened — the service classes need broad access to the object's fields, so the fields end up effectively public.

**The fix:** put behaviour next to state. `order.cancel()` lives on `Order`. `order.add_item(item)` lives on `Order`. The service exists when an operation legitimately *spans* multiple aggregates — `transfer(from_account, to_account, amount)` belongs on a `TransferService` because no single account "owns" it. But single-object operations belong on the object.

**The Domain-Driven Design vocabulary:** "rich domain models" describe objects with behaviour; "anemic models" describe objects without. Eric Evans' *Domain-Driven Design* book made the distinction famous and convinced a generation of Java developers to push behaviour back into their entities. Kotlin's data classes plus extension functions, and Python's dataclasses plus methods, make rich models genuinely lightweight.


### Example — pushing behaviour onto the model


In [ ]:
from dataclasses import dataclass, field


# Anemic — Order is a data bag, OrderService owns every operation.
@dataclass
class OrderAnemic:
    id: int
    items: list[dict]
    status: str


class OrderServiceAnemic:
    # All behaviour lives here, away from the data it operates on.
    def add_item(self, order: OrderAnemic, item: dict) -> None:
        if order.status != "draft":
            raise ValueError("can only add items to a draft order")
        order.items.append(item)

    def total(self, order: OrderAnemic) -> float:
        return sum(i["price"] * i["qty"] for i in order.items)

    def submit(self, order: OrderAnemic) -> None:
        if not order.items:
            raise ValueError("cannot submit empty order")
        order.status = "submitted"


# Rich — behaviour lives next to state. The order knows its own rules.
@dataclass
class Order:
    id: int
    items: list[dict] = field(default_factory=list)
    status: str = "draft"

    def add_item(self, item: dict) -> None:
        if self.status != "draft":
            raise ValueError("can only add items to a draft order")
        self.items.append(item)

    def total(self) -> float:
        return sum(i["price"] * i["qty"] for i in self.items)

    def submit(self) -> None:
        if not self.items:
            raise ValueError("cannot submit empty order")
        self.status = "submitted"


order = Order(id=1)
order.add_item({"name": "widget", "price": 10.0, "qty": 3})
order.add_item({"name": "gizmo",  "price": 25.0, "qty": 1})
print(order.total())   # 55.0
order.submit()
print(order.status)    # submitted


The rich version has the validation rules ("only draft orders can be modified", "cannot submit empty") right next to the data they protect. The anemic version forces the same rules to live in a separate service, which means they can be bypassed by any code that mutates `order.items` or `order.status` directly. Encapsulation is back where it belongs.


## Anti-pattern 4 — Pattern Fever

**The smell:** a codebase where every concept is a pattern. The simplest function is wrapped in a Strategy. Every singleton has a Factory. Every collection has a Decorator. New developers spend their first month learning the codebase's pattern vocabulary before they can read business logic.

**Why it happens:** patterns feel like architecture. Applying them feels like senior engineering. Senior engineers under-appreciate that the code's *vocabulary footprint* is a tax on every reader — every named pattern is a concept the reader has to recognize before they can read the actual logic underneath.

**The cost:** the cost is not in the patterns themselves but in their *combination*. A Singleton-Factory-Builder-Decorator chain wrapping a five-line operation is three levels of indirection too many. Reading the code means traversing the indirection layers to find the business logic, which is buried at the bottom.

**The fix:** delete the pattern when the smell it was meant to address either never appeared or has gone away. Patterns are reversible — a Strategy with one concrete implementation can become a direct function call. A Singleton can become a passed parameter. A Decorator with one capability can become a method on the wrapped class.

**The rule of three** is the most useful antidote. The first time you see a shape, write the simplest code that solves it. The second time, consider whether you have a pattern. The third time, apply the pattern. Patterns applied on the second occurrence are speculative; patterns applied on the third are responsive to real evidence.


## Anti-pattern 5 — Lava Layers

**The smell:** the codebase has visible geological strata of past architectural decisions. The old controllers call into the new service layer, which calls into the old facade, which calls into the new repository, which calls into the old data access object, which still has its own connection management. Each layer was added when the team decided to "do it right this time," but the old layer was never removed.

**Why it happens:** large refactors are hard, so teams pile new abstractions on top of old ones instead of replacing them. Each refactor is a partial migration. The deprecated layers remain because removing them requires updating call sites the team can't afford to revisit.

**The cost:** every read of the code traverses layers that exist only as fossils of past decisions. Performance suffers. Onboarding suffers — new engineers have to learn three architectural eras to ship a single feature.

**The fix:** *finish* refactors. When introducing a new pattern, accept that the change is not done until the old one is removed. If the migration cannot be completed in one PR, the change is too large; reduce its scope. Better to leave the old layer untouched than to add a half-migrated new layer on top.

**The discipline this requires:** treat architecture changes as atomic. A new abstraction is not merged until the *old* one is gone or has a credible deletion plan in the next sprint. "We'll clean up the old layer later" is the sentence that creates lava layers.


## Anti-pattern 6 — Premature Generalization

**The smell:** abstractions designed for use cases that do not yet exist. A `PaymentProcessor` interface with one implementation. A `NotificationStrategy` that always sends email. A `Repository<T>` generic when there is only one entity type. Configuration tables that no one configures.

**Why it happens:** "we might need to swap implementations later." This sentence is almost always wrong. The actual second implementation, if it ever arrives, has requirements the speculative abstraction did not anticipate, so the abstraction has to be rewritten anyway. The interface ends up shaped by the original implementation, not by the eventual diversity.

**The cost:** abstractions cost. They split code across more files, add indirection in stack traces, force callers to learn the abstract shape before the concrete one. A speculative abstraction pays the cost without the benefit.

**The fix:** apply YAGNI — *You Aren't Gonna Need It*. Build the concrete implementation. When a *second* concrete implementation arrives with real requirements, refactor to introduce the abstraction. The shape will fit both because both shaped it.

**The exception:** abstractions at the *boundary* of your system — where you talk to a database, a third-party API, a file system — are often worth introducing early. Not because you'll swap them, but because the boundary is where tests need to substitute fakes. A "premature" repository interface that supports an in-memory implementation for testing earns its keep on day one.


## Code smells that suggest a specific pattern

A short field guide — when *this* smell appears, *this* pattern is the usual answer.

| Smell | Pattern |
|---|---|
| A long `if/elif` chain on a type tag (`if customer_type == "vip" ...`) | **Strategy** or **State** |
| A long `if/elif` chain on a value range, where the cases will grow | **Chain of Responsibility** (first-handler-wins) |
| A class with too many constructor parameters, most optional | **Builder** |
| Duplicate code in subclasses that differ only in a step or two | **Template Method** |
| Several classes that change together when one changes | **Mediator** or **Observer** |
| Code that says "if this is null, do X; otherwise do Y" everywhere | **Null Object** |
| Business logic that needs the same predicate in several places | **Specification** |
| Tests that need a real database / network / file system to run | **Repository** (with an in-memory implementation) |
| Code that says "I want to add a step before/after this method without changing it" | **Decorator** |
| A class that does five visibly different things | **Single Responsibility** + decomposition; possibly **Facade** |
| A subsystem with many entry points the rest of the system uses inconsistently | **Facade** |
| Data structures the same shape is walked many different ways | **Visitor** (or `sealed` + `when`) |
| State that needs to be captured and restored | **Memento** |
| An operation that needs to be queued, logged, or undone | **Command** |

The key word is *suggests*. Code smells point toward patterns; they don't mandate them. A smell with only one example doesn't justify a pattern. Wait for the rule of three.


## A decision framework for choosing patterns

A four-question checklist for whether to introduce a pattern:

1. **Has the smell appeared at least three times in the codebase?** If no, don't introduce the pattern yet. Write straightforward code. Wait for evidence.
2. **Does the pattern's *intent* match the problem?** Not the structure — the intent. Strategy is for "algorithm varies at runtime," not for "we have several classes with one method each."
3. **Is the structure smaller than the problem?** A `SingletonFactoryBuilder` to construct one logger is not architecture; it's paperwork. If the pattern's footprint is bigger than the problem it solves, reconsider.
4. **Can someone unfamiliar with the codebase read it?** If recognizing the pattern is a prerequisite for understanding the business logic, the pattern is obscuring the domain. The same logic written directly should be at least as readable.

If all four answers come back positive, introduce the pattern. If one answer is uncertain, write the straightforward code and come back when more evidence accrues.


## When to delete a pattern

Patterns are not permanent. A pattern that earned its keep in 2022 may have outlived the requirements by 2025. Five signs a pattern should be deleted:

- **The variation it abstracted over has collapsed to one case.** A Strategy with only one concrete implementation, after the others have been removed, is dead weight. Inline the remaining strategy.
- **The smell it addressed has gone away.** A Repository that was introduced for testability is no longer needed if the production code now talks directly to an in-memory store. Remove the interface and let the concrete type be the type.
- **The pattern's structure is fighting newer language features.** A custom Iterator class introduced in 2010 may now be one line as a generator. A bespoke Visitor in Kotlin may now be a `when` expression with sealed types.
- **No one remembers why the pattern is there.** Code archeology turns up no commit message, no design doc, no comment explaining the choice. The pattern is cargo culted at this point and should be evaluated as if it were being introduced new.
- **The pattern is a maintenance burden.** Every change requires touching the interface, the abstract class, and three implementations. If the cost of carrying the pattern exceeds the cost of inlining it, inline it.

Deleting a pattern is just as legitimate an engineering activity as introducing one. The codebase belongs to the engineers maintaining it now, not to the engineers who wrote it. If the pattern is no longer earning its keep, remove it.


## The throughline

A useful summary of seven notebooks in one sentence: **patterns are vocabulary for design intent, not a checklist of structures to apply.**

The vocabulary is the durable contribution of the Gang of Four catalog. "Strategy" still communicates more than "callable parameter," even when the implementation is one line. The structural ceremony around each pattern is *negotiable* — Python's first-class functions, Kotlin's data classes and `sealed` types, and modern language features generally have collapsed several patterns into language constructs. The intent survives the collapse.

The honest rule of thumb has not changed in three decades: start with the simplest code that works, refactor toward a pattern when a real smell appears, name the pattern explicitly so the team's vocabulary stays sharp, and delete patterns that no longer earn their keep. Patterns are tools, not goals.
